# 🎙️ LinguaNewTab — Kokoro TTS Audio GeneratorNotebook này sinh ra file MP3 phát âm chuẩn cho **64 pinyin phonemes** (21 âm đầu + 38 âm vận + 5 thanh điệu) dùng **Kokoro TTS** voice `zf_xiaoxiao` (standard Mandarin).**Workflow:**1. ▶️ Cell 2 — Install Kokoro + dependencies (~2 phút)2. ▶️ Cell 3 — Define phonemes + syllables3. ▶️ Cell 4 — Test sample 10 file để nghe trước4. ▶️ Cell 5 — Generate full 64 phonemes + zip download**Chi phí:** $0 — Kokoro là open-source Apache 2.0, chạy free trên Colab GPU.**Voice:** `zf_xiaoxiao` — female, chuẩn Bắc Kinh (xác nhận bởi native Chinese speaker trên HuggingFace).---⚠️ **Quan trọng**: Chọn Runtime → Change runtime type → **GPU (T4)** trước khi chạy để nhanh hơn ~20×. Nếu không có GPU cũng OK, chỉ lâu hơn (~5 phút cho 64 files).

## 📦 Cell 1 — Install KokoroClick ▶️ bên trái để chạy cell này. Chờ ~2 phút.

In [ ]:
# Install kokoro-onnx (lightweight ONNX version, không cần PyTorch nặng)!pip install -q kokoro-onnx soundfile# Download Kokoro model (int8 quantized, 88MB — chất lượng gần bằng f32 nhưng tải nhanh 3.5×)!wget -q https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.int8.onnx -O kokoro-v1.0.onnx# Download voice style vectors (~26 voices, bao gồm zf_xiaoxiao, zm_yunxi, ...)!wget -q https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.binprint("✅ Install xong. Kokoro ready.")import osprint(f"Model size: {os.path.getsize('kokoro-v1.0.onnx') / 1e6:.1f} MB")print(f"Voices size: {os.path.getsize('voices-v1.0.bin') / 1e6:.1f} MB")

## 🔤 Cell 2 — Define 64 phonemesDictionary này chứa mapping từ **phoneme ID** (match với data trong `src/data/zh/pinyin.ts`) → **syllable** sẽ được đọc.Syllable được thiết kế theo pedagogy chuẩn (giống Yoyochinese/Pleco):- **Initials** (âm đầu): đi với nguyên âm tone 1 để nghe rõ (vd: `b` → `bō`)- **Finals** (âm vận): đứng riêng hoặc prefix với `y`/`w` theo pinyin rule (vd: `ia` → `yā`)- **Tones** (thanh điệu): 5 biến thể của từ `ma`

In [ ]:
import jsonPHONEMES = {    # ————— 21 Initials (âm đầu) —————    "zh_init_b":  "bō",   "zh_init_p":  "pō",   "zh_init_m":  "mō",   "zh_init_f":  "fō",    "zh_init_d":  "dē",   "zh_init_t":  "tē",   "zh_init_n":  "nē",   "zh_init_l":  "lē",    "zh_init_g":  "gē",   "zh_init_k":  "kē",   "zh_init_h":  "hē",    "zh_init_j":  "jī",   "zh_init_q":  "qī",   "zh_init_x":  "xī",    "zh_init_zh": "zhī",  "zh_init_ch": "chī",  "zh_init_sh": "shī",  "zh_init_r":  "rī",    "zh_init_z":  "zī",   "zh_init_c":  "cī",   "zh_init_s":  "sī",    # ————— 38 Finals (âm vận) —————    # Đơn âm    "zh_fin_a": "ā",  "zh_fin_o": "ō",  "zh_fin_e": "ē",    "zh_fin_i": "yī", "zh_fin_u": "wū", "zh_fin_u_umlaut": "yū",    # Phức âm 2 chữ    "zh_fin_ai": "āi",  "zh_fin_ei": "ēi",  "zh_fin_ao": "āo",  "zh_fin_ou": "ōu",    "zh_fin_an": "ān",  "zh_fin_en": "ēn",  "zh_fin_ang": "āng", "zh_fin_eng": "ēng",    "zh_fin_ing": "yīng", "zh_fin_ong": "dōng",    # Với i    "zh_fin_ia": "yā",   "zh_fin_ie": "yē",   "zh_fin_iao": "yāo", "zh_fin_iou": "yōu",    "zh_fin_ian": "yān", "zh_fin_in": "yīn",  "zh_fin_iang": "yāng", "zh_fin_iong": "yōng",    # Với u    "zh_fin_ua": "wā",   "zh_fin_uo": "wō",   "zh_fin_uai": "wāi", "zh_fin_uei": "wēi",    "zh_fin_uan": "wān", "zh_fin_uen": "wēn", "zh_fin_uang": "wāng", "zh_fin_ueng": "wēng",    # Với ü    "zh_fin_ue": "yuē",     "zh_fin_uan_u": "yuān", "zh_fin_un_u": "yūn",    # Đặc biệt    "zh_fin_er": "ēr",    "zh_fin_i_retroflex": "zhī",  # -i sau zh/ch/sh/r    "zh_fin_i_apical":    "zī",    # -i sau z/c/s    # ————— 5 Tones (thanh điệu) —————    "zh_tone_1": "mā",  "zh_tone_2": "má",  "zh_tone_3": "mǎ",    "zh_tone_4": "mà",  "zh_tone_neutral": "ma",}print(f"✅ Defined {len(PHONEMES)} phonemes")initials = sum(1 for k in PHONEMES if k.startswith('zh_init_'))finals = sum(1 for k in PHONEMES if k.startswith('zh_fin_'))tones = sum(1 for k in PHONEMES if k.startswith('zh_tone_'))print(f"   {initials} initials + {finals} finals + {tones} tones")

## 🎧 Cell 3 — Test 10 samples trướcGenerate 10 file test để Jason nghe voice `zf_xiaoxiao` trước khi chạy full batch.Chọn mix đa dạng: các initial hiếm (zh/r), final phức tạp (üe/iang), và cả 5 tones.

In [ ]:
from kokoro_onnx import Kokoroimport soundfile as sfimport numpy as npfrom IPython.display import Audio, displayimport os# Init Kokoro với model vừa downloadkokoro = Kokoro("kokoro-v1.0.onnx", "voices-v1.0.bin")# 10 sample phonemes đại diệnSAMPLES = [    "zh_init_b",    # initial cơ bản    "zh_init_zh",   # retroflex khó    "zh_init_r",    # âm đặc biệt    "zh_fin_a",     # final đơn    "zh_fin_ue",    # final với ü    "zh_fin_iang",  # final phức    "zh_tone_1",    # 4 thanh điệu + trung    "zh_tone_2",    "zh_tone_3",    "zh_tone_4",]os.makedirs("samples", exist_ok=True)print("🎙️ Generating samples với voice zf_xiaoxiao...\n")for pid in SAMPLES:    syllable = PHONEMES[pid]    # speed=0.85 để phonemes đọc chậm rõ hơn (pedagogy)    samples, sample_rate = kokoro.create(syllable, voice="zf_xiaoxiao", speed=0.85, lang="cmn")    out_path = f"samples/{pid}.wav"    sf.write(out_path, samples, sample_rate)    print(f"  ✓ {pid:25s} ({syllable})")print("\n🎧 Play từng sample bên dưới (click ▶️):")for pid in SAMPLES:    print(f"\n--- {pid} ({PHONEMES[pid]}) ---")    display(Audio(f"samples/{pid}.wav", autoplay=False))

## 🚀 Cell 4 — Generate full 64 phonemes + zipNếu Jason ưng voice sample, chạy cell này để generate full batch.**Output:**- 64 file MP3 trong `public/audio/zh/{initials,finals,tones}/`- `manifest.json` — mapping từ phoneme ID → filepath- `lingua-audio-zh.zip` — tất cả files đã zip sẵn, download về và extract vào project**Time:** ~30s trên GPU T4, ~3 phút trên CPU.

In [ ]:
import os
import shutil
import json
import subprocess
from datetime import datetime
from pathlib import Path

# Fresh output structure (matches src/services/audioService.ts expectations)
OUTPUT_ROOT = Path("public/audio/zh")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
for sub in ["initials", "finals", "tones"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

def phoneme_subdir(pid: str) -> str:
    if pid.startswith("zh_init_"): return "initials"
    if pid.startswith("zh_fin_"):  return "finals"
    if pid.startswith("zh_tone_"): return "tones"
    raise ValueError(f"Unknown phoneme type: {pid}")

# audioService expects: { generated_at, voices: {female, male}, speed, total_files, entries: {key: {female, male}} }
entries = {}

print(f"🚀 Generating {len(PHONEMES)} phonemes...\n")

for i, (pid, syllable) in enumerate(PHONEMES.items(), 1):
    subdir = phoneme_subdir(pid)
    wav_path = OUTPUT_ROOT / subdir / f"{pid}.wav"
    mp3_path = OUTPUT_ROOT / subdir / f"{pid}.mp3"

    # Generate WAV
    samples, sample_rate = kokoro.create(syllable, voice="zf_xiaoxiao", speed=0.85, lang="cmn")
    sf.write(str(wav_path), samples, sample_rate)

    # Convert to MP3 (smaller file size, ~10KB vs ~50KB WAV)
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(wav_path), "-codec:a", "libmp3lame",
         "-b:a", "64k", "-ac", "1", str(mp3_path)],
        capture_output=True, check=True
    )
    wav_path.unlink()

    # Schema compatible with audioService.resolveManifestKey()
    key = f"{subdir}/{pid}"
    rel_path = f"{subdir}/{pid}.mp3"
    entries[key] = {"female": rel_path}

    print(f"  [{i:2d}/{len(PHONEMES)}] {pid:25s} → {mp3_path.name} ({mp3_path.stat().st_size / 1024:.1f} KB)")

# Write manifest in audioService-compatible format
manifest = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "voices": {"female": "zf_xiaoxiao", "male": ""},
    "speed": 0.85,
    "total_files": len(entries),
    "entries": entries,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

total_bytes = sum((OUTPUT_ROOT / v["female"]).stat().st_size for v in entries.values())
print(f"\n✅ Done. Total: {total_bytes / 1024:.1f} KB ({len(entries)} files)")
print(f"   Manifest: {manifest_path}")

In [ ]:
# Zip everything for easy downloadimport shutilfrom google.colab import filesshutil.make_archive("lingua-audio-zh", "zip", "public/audio/zh")zip_size = os.path.getsize("lingua-audio-zh.zip") / 1024print(f"📦 lingua-audio-zh.zip ({zip_size:.1f} KB)")print("\nBắt đầu download...")files.download("lingua-audio-zh.zip")

## ✅ Xong! Next steps1. **Extract zip** vào thư mục project: `public/audio/zh/` (đã có sẵn structure)2. **Báo Claude** code xong Phase A — mình sẽ build Phase B (runtime integration trong extension):   - `src/services/audioService.ts` — load/play MP3 từ bundle   - Update `TabPhonetics.tsx` click handler   - Update `Flashcard.tsx` TTS button   - Graceful fallback về Web Speech API nếu file không tồn tại**Sau Phase A+B xong**, nếu Jason ưng chất lượng, turn tiếp mình có thể:- Generate thêm 150 HSK1 words (cùng workflow này, đổi PHONEMES = HSK1_WORDS)- Generate 500 Oxford English words với voice `af_sarah`---**Troubleshooting:**- **GPU OOM**: Runtime → Disconnect → Reconnect với GPU fresh- **Voice nghe không ưng**: thay `zf_xiaoxiao` bằng `zf_xiaoyi` (trẻ hơn) hoặc `zm_yunxi` (male)- **Syllable đọc sai**: edit PHONEMES dict trong Cell 2, re-run Cell 4 + 5